# Import Necessary Packages

In [ ]:
import os
import string
import re
import nltk
import pandas as pd
import seaborn as sns
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from textblob import Word
from unidecode import unidecode
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MaxAbsScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from textblob import TextBlob

In [ ]:
tqdm.pandas()

In [ ]:
# setting LOKY_MAX_CPU_COUNT to the number of cores you want to use
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

# Loading the data

In [ ]:
train_df = pd.read_csv(os.path.join(".","data","training_data.csv"))
test_df = pd.read_csv(os.path.join(".","data","testing_data.csv"))

In [ ]:
train_df.head(3)

# EDA Checklist

In [ ]:
print(f'Training shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

In [ ]:
train_df.describe()

In [ ]:
train_df.isnull().sum()

In [ ]:
sns.countplot(data=train_df,x="Label")

In [ ]:
train_df["Date"] = pd.to_datetime(train_df["Date"])

In [ ]:
train_df["langth"] = train_df["reviews"].apply(len)

# Text Cleaning & Preprocessing

In [ ]:
def handleRepetitive(sentence):
    rx = re.compile(r'([^\W\d_])\1{2,}')
    return re.sub(r'[^\W\d_]+', lambda x: Word(rx.sub(r'\1\1', x.group())).correct() if rx.search(x.group()) else x.group(), sentence)

def replaceHomoglyphs(text):
    return unidecode(text)

def remove_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '[URL]', text)

def remove_html(text):
    return BeautifulSoup(text, 'html.parser').get_text()

def remove_emails(text):
    canonical_email = re.sub(r'([a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+)','[EMAIL]', text)
    return canonical_email

def convertLowercase(text):
    return text.lower()

def remove_numbers(text):
    return re.sub(r'\d+', '[NUMBER]', text)

stop_words = set(nltk.corpus.stopwords.words("english"))
def removeStopWords(text):
    text = [word for word in text.split() if word not in stop_words]
    return ' '.join(text)

spetial_chars = string.punctuation
escaped_chars = [re.escape(c) for c in spetial_chars]
spetial_chars_regex = re.compile(f"({'|'.join(escaped_chars)})")

def remove_punctuation(text):
    return re.sub(spetial_chars_regex," ",text)

stemmer = nltk.stem.SnowballStemmer(language="english")

def stem(text):
    
    text = nltk.word_tokenize(text, language='english')
        
    text = [stemmer.stem(word) for word in text]
    
    return ' '.join(text)